# Think Like Karpathy

通过 git 历史学 [@karpathy](https://github.com/karpathy) 2025–2026 的 4 个开源 repo：

| Repo | 周期 | commits | 一句话 |
|---|---|---|---|
| **nanochat** | 2025-10 → 2026-04 | 380 | 单节点 + 可读代码 · 6 个月把 time-to-GPT-2 从 3.04h 压到 1.65h |
| **autoresearch** | 2026-03 (3 周) | 35 | 让 agent 过夜跑实验 · 人改 md / agent 改 py 的分权宪法 |
| **jobs** | 2026-03 (2 天) | 11 | BLS 342 职业 treemap · LLM 可换上色层 |
| **hn-time-capsule** | 2025-12 (36 小时) | 15 | 一次 vibe code 的 LLM 应用工程教科书 |

本 notebook 是**可复现的研究工具**——clone 上述 4 个 repo 到同一父目录后，从头到尾运行即可重新采样所有证据，或修改查询来验证你自己的假设。

配套文档：
- [`CONCLUSIONS.md`](CONCLUSIONS.md) — 跨 repo 综合结论（6 章）
- [`INSIGHTS.md`](INSIGHTS.md) — 深度洞察（反常识 8 条 + 思维模式 12 条）
- [`reports/*.html`](reports/) — 4 份独立深度报告（按横纵分析法结构）
- [`CHANGELOG.md`](CHANGELOG.md) — 研究过程日志

## 0. 环境准备

**前置**：在同一父目录 clone 以下 4 个 repo

```bash
mkdir auto-research && cd auto-research
git clone https://github.com/karpathy/nanochat
git clone https://github.com/karpathy/autoresearch
git clone https://github.com/karpathy/jobs
git clone https://github.com/karpathy/hn-time-capsule
```

然后修改下一个 cell 的 `REPOS_ROOT` 指向该父目录。

In [ ]:
import subprocess
from pathlib import Path
from collections import Counter

# 修改成你本地 clone 的父目录
REPOS_ROOT = Path.home() / 'auto-research'
REPOS = ['nanochat', 'autoresearch', 'jobs', 'hn-time-capsule']

def git(repo, *args, check=True):
    """Run a git command in the given repo and return stdout."""
    cmd = ['git', '-C', str(REPOS_ROOT / repo), *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f'git failed: {cmd}\n{r.stderr}')
    return r.stdout

# Sanity check: every repo present?
for r in REPOS:
    p = REPOS_ROOT / r
    assert p.is_dir(), f'Missing repo: {p} (clone it first)'
print('All repos found in', REPOS_ROOT)

## 1. Repo 规模快照

In [ ]:
print(f'{"repo":<18} {"commits":>8} {"start":>12} {"end":>12} {"days":>6} {"authors":>8}')
print('-' * 70)
for repo in REPOS:
    n = int(git(repo, 'rev-list', '--count', 'HEAD').strip())
    dates = git(repo, 'log', '--pretty=format:%ad', '--date=short').strip().splitlines()
    authors = set(git(repo, 'log', '--pretty=format:%an').strip().splitlines())
    start, end = dates[-1], dates[0]
    from datetime import date
    d_start = date.fromisoformat(start); d_end = date.fromisoformat(end)
    print(f'{repo:<18} {n:>8} {start:>12} {end:>12} {(d_end - d_start).days:>6} {len(authors):>8}')

**观察**：hn-time-capsule 1 天跨度/15 commits 是典型 vibe code；nanochat 6 个月/380 commits/70+ 贡献者是认真开源项目；autoresearch 20 天/35 commits/10 贡献者是小而快的样板；jobs 1 天/11 commits/单作者介于 vibe 和样板之间。

## 2. 热点文件（top 8）

In [ ]:
for repo in REPOS:
    names = git(repo, 'log', '--name-only', '--pretty=format:').strip().splitlines()
    names = [n for n in names if n]
    top = Counter(names).most_common(8)
    print(f'\n### {repo}')
    for fname, cnt in top:
        bar = '█' * cnt
        print(f'  {cnt:>3}  {fname}')

**读这张表的要点**：
- 修改最频繁的文件**不是代码，是 README / program.md / LOG.md** —— "上下文工程"是 Karpathy 仓库的第一等公民。
- nanochat 的 `scripts/base_train.py` (56 次) 是 leaderboard 每次破纪录的落脚点。
- autoresearch 的 `program.md` (9 次) 和 `train.py` (5 次) 形成"人改 md / agent 改 py"的分权证据。

## 3. 按月 commit 分布（看演化节奏）

In [ ]:
for repo in REPOS:
    months = git(repo, 'log', '--pretty=format:%ad', '--date=format:%Y-%m').strip().splitlines()
    dist = Counter(months)
    print(f'\n### {repo}')
    for m in sorted(dist):
        bar = '█' * min(dist[m], 60)
        print(f'  {m}  {dist[m]:>3}  {bar}')

**双峰**（nanochat 有 10 月 + 1 月两个峰）= 发布冲刺 + 科研觉醒。单峰 + 长尾（autoresearch）= 密集首周 + 稳定维护。集中（jobs / hn-time-capsule）= vibe code。

## 4. autoresearch · S1 的首日回撤（spawn.sh 之死）

Initial commit (`b11d6f2`) 包含一个 248 行的 `spawn.sh`（多-agent 编排），但**同一天就被删**——这是整个 repo 最重要的设计决策。

In [ ]:
# Day-0 timeline of autoresearch
log = git('autoresearch', 'log', '--date=format:%Y-%m-%d %H:%M', '--pretty=format:%h | %ad | %s')
lines = log.strip().splitlines()
day0 = [l for l in lines if '2026-03-06' in l]
for l in reversed(day0):  # chronological
    print(l)

In [ ]:
# Verify spawn.sh deletion
print(git('autoresearch', 'log', '--all', '--diff-filter=D', '--pretty=format:%h %ad %s', '--date=short', '--', 'spawn.sh'))

## 5. autoresearch · agent runtime robustness 是真正的难点

In [ ]:
# Count 'agent-robustness' commits by keyword
keywords = ['NaN', 'traceback', 'infinite loop', 'fallback', 'fast-fail', 'download-workers']
all_log = git('autoresearch', 'log', '--pretty=format:%h %ad %s', '--date=short')
for kw in keywords:
    hits = [l for l in all_log.splitlines() if kw.lower() in l.lower()]
    if hits:
        print(f'\n--- {kw} ---')
        for l in hits:
            print(' ', l)

**观察**：6 条"代理呆坐兜底"的 commits 几乎全在首 5 天内。两个独立社区贡献者在同一周各自修了 NaN fast-fail（haosenwang1018 on 3/9 + 另一位 Contributor on 3/11），说明这是夜跑最高频的雷。

## 6. hn-time-capsule · 分钟级 vibe code 节奏

In [ ]:
# Minute-resolution timeline — compute gaps
from datetime import datetime
log = git('hn-time-capsule', 'log', '--reverse', '--date=format:%Y-%m-%d %H:%M', '--pretty=format:%h|%ad|%s')
rows = [l.split('|', 2) for l in log.strip().splitlines()]
prev = None
for h, d, s in rows:
    ts = datetime.strptime(d, '%Y-%m-%d %H:%M')
    gap = f'+{int((ts - prev).total_seconds() / 60):>4}m' if prev else '   start'
    print(f'{h}  {d}  {gap}  {s}')
    prev = ts

**学到的**：两次坐下——12/9 下午 6 小时 10 commits（平均 37 分钟/commit，中位 < 10 分钟），12/10 早上 1.5 小时打磨 + 发布。注意 `+240m` 的大间隔（15:27 → 19:32）发生后紧跟 `add retry for 403`——真正跑一次才暴露 HN 的反爬。

## 7. jobs · `1cca284` README 定位反转（核心事件）

In [ ]:
# The pivotal commit where "AI Exposure Analysis" → "BLS Data Explorer"
print(git('jobs', 'show', '1cca284', '--stat'))
print('---')
diff = git('jobs', 'show', '1cca284', '--format=', '--', 'README.md')
# Show only first 40 lines of diff for compactness
print('\n'.join(diff.splitlines()[:40]))

**这一条 commit** 做了 6 处协同改动，全部稀释 "AI Exposure" 作为主叙事：
1. 标题 "AI Exposure of the US Job Market" → "US Job Market"
2. 加入 "This is **not** a report, a paper, or a serious economic publication"
3. AI Exposure 降级为 4 个可视层之一
4. 新增 "What AI Exposure is **NOT**" 免责段
5. 从 README 删除 "Average exposure 5.3/10"（移入 `score.py`）
6. 可视化说明由单色谱变为 4 layers

→ 这是学习"如何主动避开 Frey-Osborne 十年媒体化陷阱"的活教材。

## 8. nanochat · Leaderboard 加速拆解

In [ ]:
# Search for leaderboard-related commits
for kw in ['leaderboard', 'autoresearch', 'climbmix', 'fp8', 'batch size']:
    hits = git('nanochat', 'log', '-i', f'--grep={kw}', '--pretty=format:%h %ad %s', '--date=short').strip()
    if hits:
        print(f'\n### keyword: "{kw}"')
        for l in hits.splitlines()[:5]:
            print(' ', l)

### 加速拆解表（按 commit 追溯）

| # | 时间 | val_bpb | CORE | 加速来自 | commit |
|---|---|---|---|---|---|
| 0 | — | — | 0.2565 | OpenAI 原版 GPT-2 2019（168h） | — |
| 1 | 3.04h | 0.74833 | 0.2585 | d24 baseline | `348fbb3` |
| 2 | 2.91h | 0.74504 | 0.2578 | d26 + **fp8** (-4.3%) | `a67eba3` |
| 3 | 2.76h | 0.74645 | 0.2602 | batch 1M tokens (-5.2%) | `2c062aa` |
| 4 | 2.02h | 0.71854 | 0.2571 | **ClimbMix** 数据换血 (-26.8%) | `324e69c` |
| 5 | 1.80h | 0.71808 | 0.2690 | **autoresearch round 1** (-10.9%) | `6ed7d1d` |
| 6 | 1.65h | 0.71800 | 0.2626 | **autoresearch round 2** (-8.3%) | `a825e63` |

**杠杆排序：数据 > fp8 精度 > 代理注入 > 批大小自动化**。数据换血一次 (26.8%) 大于所有架构优化累加。

## 9. nanochat · 贡献者结构

In [ ]:
authors = git('nanochat', 'log', '--pretty=format:%an').strip().splitlines()
top = Counter(authors).most_common(10)
total = len(authors)
for a, n in top:
    bar = '█' * min(int(n * 60 / total), 60)
    pct = n * 100 / total
    print(f'{a:<30} {n:>4} ({pct:5.1f}%)  {bar}')

**观察**：Karpathy + 'Andrej' 合并占 ≈65%。**svlandeg** 已长成第二维护者（40 commits，覆盖 bug fix / CI / typo / CPU 兼容）。其他贡献者长尾——这是一个健康但 Karpathy-centric 的模型。

## 10. 跨 repo · 5 个共现模式（证据驱动）

In [ ]:
# Pattern 1: initial commit is huge (private-dev → public-drop)
print('### Pattern 1: Shadow-drive → Public-drop (initial commit size)')
for repo in REPOS:
    # first commit == earliest commit (no parents)
    first = git(repo, 'rev-list', '--max-parents=0', 'HEAD').strip().splitlines()[0]
    # total insertions in first commit
    numstat = git(repo, 'show', '--numstat', '--format=', first).strip()
    ins = sum(int(x.split('\t')[0]) for x in numstat.splitlines() if x.split('\t')[0].isdigit())
    files = len([1 for x in numstat.splitlines() if x])
    print(f'  {repo:<18} {first[:7]}  {files:>4} files  {ins:>8} insertions')

→ 4/4 initial commit 都是完整链路 push（非 MVP）。**"ship early" 的反面**——Karpathy 选择让公开第一眼就看到能跑通的全链路。

In [ ]:
# Pattern 2: time-to-first-community-PR distinguishes serious vs. vibe
print('### Pattern 2: 第一条社区 PR 的嗅觉')
for repo in REPOS:
    first_commit = git(repo, 'rev-list', '--max-parents=0', 'HEAD').strip().splitlines()[0]
    first_date = git(repo, 'show', '-s', '--format=%ad', '--date=short', first_commit).strip()
    # find first commit from a non-Karpathy author
    log = git(repo, 'log', '--pretty=format:%h|%ad|%an|%s', '--date=short').strip().splitlines()
    community = [l for l in log if 'arpathy' not in l.split('|')[2].lower() and 'andrej' not in l.split('|')[2].lower()]
    if community:
        first_comm = community[-1]  # oldest
        _, d, a, s = first_comm.split('|', 3)
        from datetime import date
        delta = (date.fromisoformat(d) - date.fromisoformat(first_date)).days
        print(f'  {repo:<18} +{delta:>3}d  {a[:20]:<20}  {s[:60]}')
    else:
        print(f'  {repo:<18}  (单作者 repo — 无社区 PR)')

→ **社区嗅觉比 README 声明更灵**：声明"长期维护"的 nanochat / autoresearch 都在 24 小时内接到第一条社区 PR；声明"不维护"的 jobs / hn-time-capsule 零外部 PR。

## 11. 扩展查询（你可以修改这些代码去验证自己的假设）

例子：

In [ ]:
# 例 1: 在 commit msg 里留情绪的频次
for repo in REPOS:
    out = git(repo, 'log', '-i', '--grep=dam\\|suffer\\|bad bad\\|oops\\|ugly\\|incredible', '--pretty=format:%h %ad %s', '--date=short').strip()
    if out:
        print(f'\n--- {repo} ---')
        for l in out.splitlines():
            print(' ', l)

In [ ]:
# 例 2: 删除比新增多的 commits（"subtract then decide"）
print('top 10 deletion-heavy commits across all 4 repos:')
deletions = []
for repo in REPOS:
    stat = git(repo, 'log', '--shortstat', '--pretty=format:@@%h|%ad|%s', '--date=short')
    blocks = stat.split('@@')[1:]
    for b in blocks:
        try:
            head, rest = b.split('\n', 1)
            h, d, s = head.split('|', 2)
            ins = del_ = 0
            import re
            m1 = re.search(r'(\d+) insertion', rest); m2 = re.search(r'(\d+) deletion', rest)
            if m1: ins = int(m1.group(1))
            if m2: del_ = int(m2.group(1))
            if del_ > ins and del_ > 50:
                deletions.append((del_ - ins, repo, h, d, s[:70]))
        except ValueError:
            continue
deletions.sort(reverse=True)
for net, repo, h, d, s in deletions[:10]:
    print(f'  -{net:<5}  {repo:<16} {h}  {d}  {s}')

## 12. 下一步阅读

1. **跨 repo 综合结论** → [`CONCLUSIONS.md`](CONCLUSIONS.md)
2. **反常识 8 条 + 思维模式 12 条** → [`INSIGHTS.md`](INSIGHTS.md)
3. **单 repo 深度报告**（横纵分析法）→ `reports/<repo>.html`
4. **研究过程** → [`CHANGELOG.md`](CHANGELOG.md)

---

**研究的 meta-原则**：
- 所有结论来源于可 `git blame` 溯源的公开信息；不猜内部决策。
- 信息缺口显式标注，不用模糊叙述填补。
- 用户视角优先引用可追溯来源（PR #编号、commit msg 原文、LOG.md 片段），而非"大家都觉得"。